In [4]:
import pandas as pd
from io import StringIO

def expand_table_with_missing_bpm(df):
    # Skip the first two rows (header and units)
    data = df.iloc[1:].copy()
    
    # Convert columns to numeric where applicable
    data['HR'] = pd.to_numeric(data['HR'], errors='coerce')
    data['Calories'] = pd.to_numeric(data['Calories'], errors='coerce')
    
    # Create a list to store expanded rows
    expanded_rows = []
    
    # Iterate through rows to interpolate missing BPM values
    for i in range(len(data) - 1):
        current_row = data.iloc[i]
        next_row = data.iloc[i + 1]
        
        current_bpm = current_row['HR']
        next_bpm = next_row['HR']
        
        # Add the current row to the expanded rows
        expanded_rows.append(current_row)
        
        # Check if there are missing BPM values
        if next_bpm - current_bpm > 1:
            missing_bpm_count = int(next_bpm - current_bpm - 1)
            calorie_diff = (next_row['Calories'] - current_row['Calories']) / (missing_bpm_count + 1)
            
            # Generate missing rows
            for j in range(1, missing_bpm_count + 1):
                interpolated_bpm = current_bpm + j
                interpolated_calories = current_row['Calories'] + calorie_diff * j
                
                # Create a new row with interpolated values
                interpolated_row = current_row.copy()
                interpolated_row['HR'] = interpolated_bpm
                # Calculate calories per second based on the interpolated value
                interpolated_row['Calories'] = interpolated_calories
                
                # Add the interpolated row to the expanded rows
                expanded_rows.append(interpolated_row)
    
    # Add the last row to the expanded rows
    last_row = data.iloc[-1].copy()
    expanded_rows.append(last_row)
    
    # Convert the list of rows back to a DataFrame
    expanded_data = pd.DataFrame(expanded_rows)

    # Add a new column for calories per minute
    expanded_data['Calories_Second'] = expanded_data['Calories'] / 60
    
    return expanded_data

def import_to_duckdb(df, table_name, db_file='workout_data.db', replace=False):
    """
    Import a pandas DataFrame into a DuckDB table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame to import into DuckDB
    table_name : str
        Name for the DuckDB table
    db_file : str, optional
        Path to DuckDB database file (default: 'workout_data.db')
    replace : bool, optional
        Whether to replace an existing table (default: False)
    
    Returns:
    --------
    duckdb.DuckDBPyConnection
        The active DuckDB connection
    """
    import duckdb
    
    # Create a DuckDB connection
    con = duckdb.connect(db_file)
    
    # Check if table exists and handle accordingly
    table_exists = con.execute(f"SELECT count(*) FROM information_schema.tables WHERE table_name='{table_name}'").fetchone()[0] > 0
    
    if table_exists:
        if replace:
            con.execute(f"DROP TABLE IF EXISTS {table_name}")
        else:
            print(f"Table '{table_name}' already exists. Set replace=True to overwrite.")
            return con
    
    # Import the DataFrame into a DuckDB table
    con.execute(f"CREATE TABLE {table_name} AS SELECT * FROM df")
    
    # Verify the import
    count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"Data imported to DuckDB table '{table_name}' with {count} rows")
    
    con.close()

In [5]:
import pandas as pd
from io import StringIO

# Read the CSV data into a DataFrame
# df = pd.read_csv(StringIO(csv_data), sep="\t")
df = pd.read_csv("./v02max_data.csv")
#print(df)

# Keep only HR and Calories columns
df = df[['HR', 'Calories']]

# Print to total count
print(f"Total rows in original DataFrame: {len(df)}")

# Expand the table
expanded_df = expand_table_with_missing_bpm(df)
print(f"Total rows in expanded DataFrame: {len(expanded_df)}")

# Set display options to show all rows and columns without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.6f}'.format)

# Find the index of the maximum HR value in expanded_df
# The reason we want to have a subset till max value is
# because after maxing out HR, after that we will have duplicates but different calory burn.
# Different calories burn because the body is cooling down.
max_hr_value = expanded_df['HR'].max()
max_hr_index = expanded_df[expanded_df['HR'] == max_hr_value].index[0]

print(f"Maximum HR value in expanded_df: {max_hr_value} at index {max_hr_index}")

# Create a subset of the expanded DataFrame from index 0 to the index of maximum HR
hr_rise_expanded_df = expanded_df.loc[:max_hr_index]
print(f"Total rows in expanded sliced (0:{max_hr_index}) DataFrame : {len(hr_rise_expanded_df)}")

# Calculate the average calorie difference b/n bpm
# TODO

# Sorting it because we can have the following sequence of HR
# e.g., 150, 151, 152, 151, 150.
# Sorting it will put all the same HR values next to each other, so the collapsing algo
# below will be able to collapse them properly.
hr_rise_expanded_df = hr_rise_expanded_df.sort_values(by='HR').reset_index(drop=True)

# Display the subset of expanded DataFrame (data up to max HR)
print("\nExpanded data from HR rise (index 0 to max HR):")
display(hr_rise_expanded_df)

# Now continue with the collapsing process
# Create a group identifier for consecutive identical HR values
hr_rise_expanded_df['group'] = (hr_rise_expanded_df['HR'] != hr_rise_expanded_df['HR'].shift()).cumsum()

# Group by both group and HR to collapse only consecutive duplicates
collapsed_df = hr_rise_expanded_df.groupby(['group', 'HR']).agg({
    'Calories': 'mean',
    'Calories_Second': 'mean'
}).reset_index().drop('group', axis=1)
print(f"Total rows in collapsed DataFrame: {len(collapsed_df)}")

# Display the collapsed DataFrame
print("\nFull collapsed DataFrame:")
display(collapsed_df)

# Write the collapsed DataFrame to DuckDB
db_dir = './hr_data/database_v2.duckdb'
import_to_duckdb(collapsed_df, 'calories_per_hr', db_dir, replace=True)

Total rows in original DataFrame: 85
Total rows in expanded DataFrame: 126
Maximum HR value in expanded_df: 185.0 at index 72
Total rows in expanded sliced (0:72) DataFrame : 114

Expanded data from HR rise (index 0 to max HR):


,HR,Calories,Calories_Second
0,96.000000,2.330000,0.038833
1,97.000000,2.485000,0.041417
2,98.000000,2.640000,0.044000
3,99.000000,2.795000,0.046583
4,100.000000,2.950000,0.049167
5,101.000000,3.105000,0.051750
6,102.000000,3.260000,0.054333
7,103.000000,3.415000,0.056917
8,104.000000,3.570000,0.059500
9,105.000000,3.725000,0.062083


Total rows in collapsed DataFrame: 90

Full collapsed DataFrame:


,HR,Calories,Calories_Second
0,96.000000,2.330000,0.038833
1,97.000000,2.485000,0.041417
2,98.000000,2.640000,0.044000
3,99.000000,2.795000,0.046583
4,100.000000,2.950000,0.049167
5,101.000000,3.105000,0.051750
6,102.000000,3.260000,0.054333
7,103.000000,3.415000,0.056917
8,104.000000,3.570000,0.059500
9,105.000000,3.725000,0.062083


Data imported to DuckDB table 'calories_per_hr' with 90 rows
